# Proyecto: Valorización energética de biomasa de palma - Magdalena
## Modelo Python/Pyomo basado en Capítulo 5 de tesis doctoral

> **Notebook reorganizado** — se eliminaron celdas de intentos intermedios/superados que existían en la versión anterior, para dejar solo la version final y correcta de cada bloque. Ver resumen de cambios que Claude explicó en el chat.

## 0. Imports generales
**Ejecutar siempre esta celda primero después de reiniciar el kernel.**

In [1]:
import pandas as pd
import numpy as np
from iapws import IAPWS97
import pyomo.environ as pyo


## 1. Limpieza — Datos_Base
Parámetros de línea base por planta, organizados por etapa de proceso. Incluye corrección de espacios invisibles (`\xa0`) que causaban valores tipo texto en Presión/Temperatura de vapor.

In [2]:
# ==========================================================
# Limpieza completa de Datos_Base (con corrección de texto/nbsp)
# ==========================================================
raw = pd.read_excel('Escenarios.xlsx', sheet_name='Datos_Base', header=None, usecols='A:H')
plantas = raw.iloc[1, 2:8].tolist()

df_datos = raw.iloc[2:].copy()
df_datos.columns = ['Parametro', 'Unidad'] + plantas
df_datos = df_datos.reset_index(drop=True)

# Identificar encabezados de etapa (sin unidad y sin ningún valor en las 6 plantas)
is_header = df_datos['Parametro'].notna() & df_datos['Unidad'].isna() & df_datos[plantas].isna().all(axis=1)
df_datos['Etapa'] = df_datos['Parametro'].where(is_header).ffill()
df_datos = df_datos[~is_header].reset_index(drop=True)

# Rellenar nombre de parámetro para las filas "huérfanas" (valor específico por tRFF)
df_datos['Parametro'] = df_datos['Parametro'].ffill()
dup_mask = df_datos.duplicated(subset=['Etapa', 'Parametro'], keep='first')
df_datos.loc[dup_mask, 'Parametro'] = df_datos.loc[dup_mask, 'Parametro'] + ' (específico por tRFF)'
df_datos.loc[dup_mask, 'Unidad'] = df_datos.loc[dup_mask, 'Unidad'].fillna('por tRFF procesado')

# Formato tidy
df_tidy = df_datos.melt(id_vars=['Parametro', 'Unidad', 'Etapa'], var_name='Planta', value_name='Valor')
df_tidy['Etapa'] = df_tidy['Etapa'].fillna('General')

# --- CORRECCIÓN: limpiar espacios invisibles (\xa0) y forzar tipo numérico ---
df_tidy['Valor'] = df_tidy['Valor'].apply(lambda v: str(v).replace('\xa0', '').strip() if isinstance(v, str) else v)
df_tidy['Valor'] = pd.to_numeric(df_tidy['Valor'], errors='coerce')

# Separar filas con dato realmente faltante (para no perderlas silenciosamente)
faltantes = df_tidy[df_tidy['Valor'].isna() & (df_tidy['Etapa'] != 'General')]
df_tidy = df_tidy.dropna(subset=['Valor']).reset_index(drop=True)

print("Filas con dato faltante en el Excel original:")
print(faltantes[['Parametro', 'Unidad', 'Etapa']].drop_duplicates())

# Corregir unidad de "Humedad de secado" (error de tecleo en el Excel original: °C -> %)
df_tidy.loc[df_tidy['Parametro'] == 'Humedad de secado', 'Unidad'] = '%'

# Agregar de vuelta "Flujo de aire de secado" como dato pendiente explícito (NaN documentado)
fila_pendiente = pd.DataFrame({
    'Parametro': ['Flujo de aire de secado'] * len(plantas),
    'Unidad': ['kgaire seco∙h-1'] * len(plantas),
    'Etapa': ['Secado de nueces'] * len(plantas),
    'Planta': plantas,
    'Valor': [np.nan] * len(plantas)
})

# Separar eficiencias reportadas (no se usan como dato de entrada - se recalculan desde cero)
es_eficiencia = df_tidy['Parametro'].str.contains('Eficiencia', case=False, na=False)
df_datos_base = df_tidy[~es_eficiencia].reset_index(drop=True)
df_datos_base = pd.concat([df_datos_base, fila_pendiente], ignore_index=True)

print("\ndf_datos_base listo:", df_datos_base.shape)

# Verificación específica: Presión/Temperatura/Flujo de vapor de Planta A (deben ser números)
verificacion = df_datos_base[
    (df_datos_base['Parametro'].isin(['Presión de vapor', 'Temperatura de vapor',
                                        'Temperatura agua de alimentación', 'Flujo de vapor'])) &
    (df_datos_base['Planta'] == 'Planta A')
]
print("\nVerificación Planta A:")
print(verificacion)


Filas con dato faltante en el Excel original:
                                            Parametro              Unidad  \
15       Eficiencia energpetica (específico por tRFF)  por tRFF procesado   
41                            Flujo de aire de secado     kgaire seco∙h-1   
55  Temperatura agua de alimentación (específico p...  por tRFF procesado   
76                                        Humedad MPD                   %   

               Etapa  
15    Esterilización  
41  Secado de nueces  
55           Caldera  
76        Desfrutado  

df_datos_base listo: (313, 5)

Verificación Planta A:
                           Parametro       Unidad    Etapa    Planta    Valor
48                    Flujo de vapor  kgvapor∙h-1  Caldera  Planta A  71984.0
49                  Presión de vapor          bar  Caldera  Planta A      3.0
50              Temperatura de vapor           °C  Caldera  Planta A    300.0
51  Temperatura agua de alimentación           °C  Caldera  Planta A     90.0


In [3]:
df_datos_base.to_csv('datos_base_limpio.csv', index=False)

## 2. Limpieza — Biomass
Propiedades fisicoquímicas, capacidad calorífica, correlación HHV ("Present study", sin término de Ash) y validación cruzada.

In [4]:
# ==========================================================
# PASO 1: Limpiar la tabla de propiedades (filas 5-15)
# ==========================================================
raw = pd.read_excel('Escenarios.xlsx', sheet_name='Biomass', header=None)

materiales = raw.iloc[4, 2:12:2].tolist()
propiedades = raw.iloc[7:15, 0].tolist()
unidades = raw.iloc[7:15, 1].ffill().tolist()

registros = []
for i, mat in enumerate(materiales):
    col_avg = 2 + i*2
    col_std = 3 + i*2
    for j, fila_excel in enumerate(range(7, 15)):
        registros.append({
            'Material': mat,
            'Propiedad': propiedades[j],
            'Unidad': unidades[j],
            'Promedio': raw.iloc[fila_excel, col_avg],
            'Desviacion_std': raw.iloc[fila_excel, col_std]
        })

df_biomasa = pd.DataFrame(registros)
print("df_biomasa listo:", df_biomasa.shape)

# ==========================================================
# PASO 2: Tabla de capacidad calorífica (de la imagen)
# ==========================================================
df_calor_especifico = pd.DataFrame({
    'Material': ['Water', 'Nut shell', 'Kernel', 'Palm oil', 'Fiber', 'EFB', 'Mud, etc.'],
    'Peso_pct': [15, 6.8, 5.2, 23.5, 14.0, 22.0, 12],
    'Cp_kJ_kgK': [4.18, 1.88, 1.59, 1.46, 1.80, 1.67, 2.22]
})

# ==========================================================
# PASO 3: Función de correlación HHV "Present study" (Eq. 7)
# ==========================================================
def hhv_present_study(C, H, O, N, S):
    """C, H, O, N, S en % base seca (dry basis). Retorna HHV en MJ/kg."""
    return 0.3443*C + 1.192*H - 0.113*O - 0.024*N + 0.093*S

# ==========================================================
# PASO 4: Validación cruzada (usa df_biomasa del Paso 1)
# ==========================================================
df_wide = df_biomasa.pivot_table(index='Material', columns='Propiedad', values='Promedio').reset_index()

df_wide['HHV_calculado_MJkg'] = df_wide.apply(
    lambda row: hhv_present_study(row['C']*100, row['H']*100, row['O']*100, row['N']*100, row['S']*100),
    axis=1
)
df_wide['HHV_reportado_MJkg'] = df_wide['HHV'] / 1000

df_wide[['Material', 'HHV_calculado_MJkg', 'HHV_reportado_MJkg']]


df_biomasa listo: (40, 5)


Propiedad,Material,HHV_calculado_MJkg,HHV_reportado_MJkg
0,Dry EFB,18.363408,18.363408
1,EFB,18.363408,18.363408
2,Fiber,19.228173,19.228173
3,PKS,19.108257,19.108257
4,Pressed EFB,18.363408,18.363408


In [5]:
df_biomasa.to_csv('biomasa_propiedades.csv', index=False)
df_calor_especifico.to_csv('biomasa_calor_especifico.csv', index=False)

## 3. Limpieza — CAPEX
Costos por área, curva de cogeneración (Malico et al. 2019), ajuste CEPCI a 2026, escalamiento de pretratamiento térmico (regla 6/10).

*(Usa `df_datos_base` de la Sección 1 — asegúrate de haber corrido esa celda primero.)*

In [6]:
# ==========================================================
# PASO 1: Limpiar el resumen por área (CAPEX, filas 14-38 = índices 13-37)
# ==========================================================
raw_capex = pd.read_excel('Escenarios.xlsx', sheet_name='CAPEX', header=None)

registros_capex = []
for r in range(13, 38):
    area = raw_capex.iloc[r, 6]         # columna G
    total = raw_capex.iloc[r, 9]        # columna J
    especifico = raw_capex.iloc[r, 10]  # columna K
    unidad = raw_capex.iloc[r, 11]      # columna L
    if pd.notna(area):
        registros_capex.append({
            'Area': area,
            'Total_USD_ref': pd.to_numeric(total, errors='coerce'),
            'Valor_especifico': pd.to_numeric(especifico, errors='coerce'),
            'Unidad_especifico': unidad
        })

df_capex = pd.DataFrame(registros_capex)
print("df_capex:")
print(df_capex)

# ==========================================================
# PASO 2: Ajuste de todos los valores a USD 2026 (CEPCI)
# ==========================================================
CEPCI_2019 = 607.5
CEPCI_2021 = 708.8
CEPCI_2026 = 873.8  # estimado, ver nota metodológica

factor_cepci_2019 = CEPCI_2026 / CEPCI_2019
factor_cepci_2021 = CEPCI_2026 / CEPCI_2021

df_capex['Total_USD_2026'] = df_capex['Total_USD_ref'] * factor_cepci_2019
df_capex['Valor_especifico_2026'] = df_capex['Valor_especifico'] * factor_cepci_2019

print(df_capex[['Area', 'Total_USD_2026', 'Valor_especifico_2026']])

# ==========================================================
# PASO 3: Curva de costo de cogeneración (Malico et al., 2019)
# ==========================================================
def capex_cogeneracion_usd_kw(capacidad_kw):
    """CAPEX específico (USD/kW) según capacidad instalada. Fuente: Malico et al. (2019). Retorna en USD 2026."""
    valor_2019 = -900.6 * np.log(capacidad_kw) + 10841
    return valor_2019 * factor_cepci_2019

print("\nEjemplo curva cogeneración (1500 kW):", capex_cogeneracion_usd_kw(1500))

# ==========================================================
# PASO 4: Escalamiento de Pretratamiento térmico (regla de los 6/10)
# ==========================================================
COSTO_REF_2021 = 1_400_000
COSTO_REF_2026 = COSTO_REF_2021 * factor_cepci_2021
CAPACIDAD_REF = 150_000
n = 0.6

def costo_pretratamiento_termico(capacidad_planta_t_ano):
    return COSTO_REF_2026 * (capacidad_planta_t_ano / CAPACIDAD_REF) ** n

print(f"\nCosto de referencia ajustado a 2026: ${COSTO_REF_2026:,.0f} USD")

fruto_procesado = df_datos_base[df_datos_base['Parametro'] == 'Fruto procesado'].set_index('Planta')['Valor']
for planta, tRFF_ano in fruto_procesado.items():
    costo = costo_pretratamiento_termico(tRFF_ano)
    print(f"{planta}: {tRFF_ano:,.0f} tRFF/año -> ${costo:,.0f} USD (2026)")


df_capex:
                                         Area  Total_USD_ref  \
0                                   Recepción   1.827019e+05   
1                 Esterilización convencional   5.842191e+05   
2                     Esterilización continua   5.760070e+05   
3                          Prensado de raquis   1.117692e+05   
4                                 Desfrutado    1.332761e+05   
5           Extracción (digestión y prensado)   2.583898e+05   
6                      Clarificación dinamica   3.505156e+05   
7                      Clarificación estatica   2.804464e+05   
8                              Almacenamiento   2.014153e+05   
9                                 Palmisteria   5.534873e+05   
10                      Caldera media presión   1.141062e+06   
11                      Turbina Contrapresiòn   2.814681e+03   
12                    Redes e infraestructura   1.072970e+05   
13            Turbina Extracción condensación            NaN   
14                Sistema gase

In [7]:
df_capex.to_csv('capex_limpio_2026.csv', index=False)

## 4. Limpieza — EmisionesCO2
Factores de emisión IPCC (Gómez et al., 2017), factor de red nacional, crédito de absorción de palma. Reconstruido desde cero (la hoja original tenía un error de factor CH4).

In [8]:
# ==========================================================
# Tabla de factores de emisión (Gómez et al., 2017 - valores IPCC por defecto)
# ==========================================================
df_factores_emision = pd.DataFrame({
    'Gas': ['CH4', 'CH4', 'N2O', 'N2O'],
    'Fuente': ['Biomasa sólida', 'Biogás', 'Biomasa sólida', 'Biogás'],
    'Factor_kg_TJ': [30, 1, 4, 0.1],
    'GWP100_kgCO2eq_kg': [21, 21, 310, 310]
})

FACTOR_CO2_DIESEL = 74100        # kg CO2/TJ (IPCC 2006, validado contra hoja original)
FACTOR_RED_NACIONAL = 0.1634     # kg CO2/kWh (163.4 gCO2/kWh - valor usado en publicación previa)
FACTOR_ABSORCION_PALMA = -135.04 # kg CO2/tRFF (crédito por absorción, cultivo de palma)

def emisiones_no_co2_biomasa(energia_TJ, fuente='Biomasa sólida'):
    """Emisiones de CH4 y N2O (no-CO2), en kg CO2-eq. El CO2 biogénico no se contabiliza (convención IPCC)."""
    fila_ch4 = df_factores_emision[(df_factores_emision['Gas']=='CH4') & (df_factores_emision['Fuente']==fuente)].iloc[0]
    fila_n2o = df_factores_emision[(df_factores_emision['Gas']=='N2O') & (df_factores_emision['Fuente']==fuente)].iloc[0]
    ch4_eq = energia_TJ * fila_ch4['Factor_kg_TJ'] * fila_ch4['GWP100_kgCO2eq_kg']
    n2o_eq = energia_TJ * fila_n2o['Factor_kg_TJ'] * fila_n2o['GWP100_kgCO2eq_kg']
    return ch4_eq + n2o_eq

def emisiones_diesel(energia_TJ):
    return energia_TJ * FACTOR_CO2_DIESEL

def emisiones_red_nacional(electricidad_kWh):
    return electricidad_kWh * FACTOR_RED_NACIONAL

def balance_neto_GEI(emisiones_proceso_kgCO2eq, tRFF_procesado):
    credito_absorcion = FACTOR_ABSORCION_PALMA * tRFF_procesado
    return emisiones_proceso_kgCO2eq + credito_absorcion

def excedente_electrico(electricidad_generada_kWh, electricidad_consumida_kWh):
    return electricidad_generada_kWh - electricidad_consumida_kWh

# Validación contra Bloque 1 (línea base) - Planta A
tj_diesel_plantaA = 0.46090413989315915
tRFF_plantaA = 153157
print("Emisiones diésel Planta A:", emisiones_diesel(tj_diesel_plantaA), "≈ 34,152.99 (valor original de la hoja)")
print("Crédito absorción Planta A:", FACTOR_ABSORCION_PALMA * tRFF_plantaA, "kg CO2")


Emisiones diésel Planta A: 34152.996766083095 ≈ 34,152.99 (valor original de la hoja)
Crédito absorción Planta A: -20682321.279999997 kg CO2


In [9]:
df_factores_emision.to_csv('factores_emision.csv', index=False)

## 5. Módulo social (Hoja2)
Indicador de viviendas/personas rurales beneficiadas por excedente eléctrico. Personas por hogar: DANE CNPV 2018 (Magdalena).

In [10]:
# ==========================================================
# Datos de demanda residencial departamental (Magdalena, 2025)
# ==========================================================
DEMANDA_RESIDENCIAL_RURAL_GWh = 30.3
DEMANDA_RESIDENCIAL_URBANA_GWh = 1209
DEMANDA_INDUSTRIAL_COMERCIO_GWh = 1190.7
DEMANDA_TOTAL_DEPARTAMENTAL_GWh = 2430  # validado: suma de las 3 anteriores

USUARIOS_RURALES = 87516
USUARIOS_URBANOS = 231790
USUARIOS_TOTAL_DANE_CNPV2018 = 319306  # validado contra fuente DANE

PERSONAS_POR_HOGAR_MAGDALENA = 3.7  # DANE, CNPV 2018 (promedio departamental)

CONSUMO_PROMEDIO_VIVIENDA_RURAL_kWh = (DEMANDA_RESIDENCIAL_RURAL_GWh * 1_000_000) / USUARIOS_RURALES
CONSUMO_PROMEDIO_VIVIENDA_URBANA_kWh = (DEMANDA_RESIDENCIAL_URBANA_GWh * 1_000_000) / USUARIOS_URBANOS

def viviendas_rurales_beneficiadas(excedente_electrico_kWh_ano):
    return excedente_electrico_kWh_ano / CONSUMO_PROMEDIO_VIVIENDA_RURAL_kWh

def personas_rurales_beneficiadas(excedente_electrico_kWh_ano):
    return viviendas_rurales_beneficiadas(excedente_electrico_kWh_ano) * PERSONAS_POR_HOGAR_MAGDALENA

def porcentaje_cobertura_rural_departamental(excedente_electrico_kWh_ano):
    return (excedente_electrico_kWh_ano / (DEMANDA_RESIDENCIAL_RURAL_GWh * 1_000_000)) * 100

def construir_tabla_impacto_social(excedentes_dict, nombre_escenario):
    """excedentes_dict: {'Planta A': excedente_kWh_ano, ...}; nombre_escenario: 'Linea_Base', 'Escenario_1_Termico', 'Escenario_2_Biogas'"""
    registros = []
    for planta, excedente in excedentes_dict.items():
        registros.append({
            'Escenario': nombre_escenario,
            'Planta': planta,
            'Excedente_kWh_ano': excedente,
            'Viviendas_rurales_beneficiadas': viviendas_rurales_beneficiadas(excedente),
            'Personas_rurales_beneficiadas': personas_rurales_beneficiadas(excedente),
            'Pct_cobertura_rural_departamental': porcentaje_cobertura_rural_departamental(excedente)
        })
    return pd.DataFrame(registros)

print(f"Consumo promedio vivienda rural: {CONSUMO_PROMEDIO_VIVIENDA_RURAL_kWh:.1f} kWh/año")
print(f"Personas por hogar (Magdalena, DANE CNPV 2018): {PERSONAS_POR_HOGAR_MAGDALENA}")


Consumo promedio vivienda rural: 346.2 kWh/año
Personas por hogar (Magdalena, DANE CNPV 2018): 3.7


In [11]:
import json
datos_sociales = {
    'demanda_rural_GWh': DEMANDA_RESIDENCIAL_RURAL_GWh,
    'demanda_urbana_GWh': DEMANDA_RESIDENCIAL_URBANA_GWh,
    'usuarios_rurales': USUARIOS_RURALES,
    'usuarios_urbanos': USUARIOS_URBANOS,
    'personas_por_hogar': PERSONAS_POR_HOGAR_MAGDALENA
}
with open('datos_sociales.json', 'w') as f:
    json.dump(datos_sociales, f, indent=2)


## 6. Limpieza — Eficiencia
Esterilización continua (escalamiento para Escenarios 1 y 2) y demanda energética por etapa de proceso (línea base).

In [12]:
CONSUMO_ESPECIFICO_ESTERILIZACION_CONTINUA = 315  # kg vapor / tRFF procesado

def vapor_esterilizacion_continua(capacidad_planta_tRFF_h):
    """Vapor requerido para esterilización continua (kg/h), escalado por capacidad de planta."""
    return CONSUMO_ESPECIFICO_ESTERILIZACION_CONTINUA * capacidad_planta_tRFF_h

capacidades = {'Planta A': 30, 'Planta B': 40, 'Planta C': 30, 'Planta D': 24, 'Planta E': 30, 'Planta F': 20}
for planta, cap in capacidades.items():
    vapor = vapor_esterilizacion_continua(cap)
    print(f"{planta}: {vapor:,.0f} kg vapor/h (esterilización continua)")


Planta A: 9,450 kg vapor/h (esterilización continua)
Planta B: 12,600 kg vapor/h (esterilización continua)
Planta C: 9,450 kg vapor/h (esterilización continua)
Planta D: 7,560 kg vapor/h (esterilización continua)
Planta E: 9,450 kg vapor/h (esterilización continua)
Planta F: 6,300 kg vapor/h (esterilización continua)


In [13]:
raw_efi = pd.read_excel('Escenarios.xlsx', sheet_name='Eficiencia', header=None)

plantas = ['Planta A', 'Planta B', 'Planta C', 'Planta D', 'Planta E', 'Planta F']
col_inicio = 13  # cada planta ocupa 2 columnas (Eléctrica, Térmica), empieza en columna N

registros_demanda = []
for r in range(5, 23):  # filas 6-23 de Excel: 'Recepción' a 'Total'
    etapa = raw_efi.iloc[r, 12]  # columna M
    if pd.isna(etapa):
        continue
    for i, planta in enumerate(plantas):
        col_e = col_inicio + i*2
        col_t = col_inicio + i*2 + 1
        valor_e = raw_efi.iloc[r, col_e]
        valor_t = raw_efi.iloc[r, col_t]
        if pd.notna(valor_e):
            registros_demanda.append({'Etapa': etapa, 'Planta': planta, 'Tipo_energia': 'Electrica', 'Valor_kW': valor_e})
        if pd.notna(valor_t):
            registros_demanda.append({'Etapa': etapa, 'Planta': planta, 'Tipo_energia': 'Termica', 'Valor_kW': valor_t})

df_demanda_por_etapa = pd.DataFrame(registros_demanda)

resumen = df_demanda_por_etapa[df_demanda_por_etapa['Etapa'].isin(['Total', 'Eficiencia global planta extractora'])]
df_demanda_por_etapa = df_demanda_por_etapa[~df_demanda_por_etapa['Etapa'].isin(['Total', 'Eficiencia global planta extractora'])].reset_index(drop=True)

print(df_demanda_por_etapa.shape)
df_demanda_por_etapa.head(20)


(103, 4)


,Etapa,Planta,Tipo_energia,Valor_kW
0,Recepción,Planta A,Electrica,1.000000
1,Recepción,Planta B,Electrica,0.170000
2,Recepción,Planta D,Electrica,0.504101
3,Recepción,Planta E,Electrica,0.200000
4,Recepción,Planta F,Electrica,0.433371
5,Esterilización,Planta A,Electrica,0.080000
6,Esterilización,Planta A,Termica,4866.736111
7,Esterilización,Planta B,Electrica,0.120000
8,Esterilización,Planta B,Termica,6156.944444
9,Esterilización,Planta C,Electrica,0.410000


In [14]:
df_demanda_por_etapa.to_csv('demanda_energia_por_etapa.csv', index=False)

### 6.1 Parámetros Tabla 10 - sustitución tecnológica

In [15]:
# ==========================================================
# Tabla 10 (tesis) - Parámetros de sustitución tecnológica de proceso
# Aplican a Escenarios 1 y 2 (reemplazan procesos convencionales de línea base)
# ==========================================================
PARAM_ESTERILIZACION_CONTINUA = {
    'consumo_vapor_kg_tRFF': 315,          # confirmado correcto (caso de estudio, hoja Eficiencia)
    'consumo_electricidad_kWh_tRFF': 1.95, # Tabla 10 tesis
    'presion_bar': 1,
    'temperatura_C': 120
}

PARAM_CLARIFICACION_DINAMICA = {
    'consumo_vapor_kg_tRFF': 0.6,       # Fernández et al. (2016)
    'consumo_electricidad_kWh_tRFF': 15,
    'consumo_agua_L_tRFF': 90
}

### 6.2 Pretratamiento mecánico de EFB — Tabla 11 (Escenario 1)
Secador rotatorio con gases de combustión de caldera. Fuente: Ocampo Batlle et al. (2020)

In [16]:
# ==========================================================
# Tabla 11 (tesis) - Pretratamiento mecánico de EFB (secador rotatorio)
# Usa gases de combustión de caldera (calor residual) - Escenario 1
# Fuente: Ocampo Batlle et al. (2020)
# ==========================================================
PARAM_PRETRATAMIENTO_MECANICO_EFB = {
    'temp_entrada_C': 25,
    'temp_salida_C': 110,
    'temp_gases_combustion_C': 215,
    'humedad_inicial_pct': 50,
    'humedad_final_pct': 15,
    'consumo_electricidad_kWh_t_agua_extraida': 19,
    'entalpia_vaporizacion_agua_kJ_kg': 2691.1
}

def agua_extraida_por_tonelada_efb(humedad_inicial_pct, humedad_final_pct, masa_efb_humedo_t=1):
    """Balance de humedad: agua extraída (t) por tonelada de EFB húmedo que entra al secador."""
    materia_seca_t = masa_efb_humedo_t * (1 - humedad_inicial_pct/100)
    masa_efb_final_t = materia_seca_t / (1 - humedad_final_pct/100)
    agua_inicial_t = masa_efb_humedo_t - materia_seca_t
    agua_final_t = masa_efb_final_t - materia_seca_t
    return agua_inicial_t - agua_final_t

agua_extraida = agua_extraida_por_tonelada_efb(50, 15)
electricidad_por_t_efb = agua_extraida * PARAM_PRETRATAMIENTO_MECANICO_EFB['consumo_electricidad_kWh_t_agua_extraida']

print(f"Agua extraída: {agua_extraida:.4f} t agua / t EFB húmedo")
print(f"Consumo eléctrico: {electricidad_por_t_efb:.2f} kWh / t EFB húmedo procesado")

Agua extraída: 0.4118 t agua / t EFB húmedo
Consumo eléctrico: 7.82 kWh / t EFB húmedo procesado


### 6.3 Parámetros definitivos — Sistema de extracción-condensación (Escenario 1)

In [17]:
# ==========================================================
# Tabla 6 (tesis) - Parámetros del sistema de cogeneración
# con turbina de extracción-condensación (Escenario 1)
# Reemplaza y consolida la Tabla 12 (valores previos, superados)
# ==========================================================
PARAM_CALDERA_ESC1 = {
    'presion_vapor_bar': 48,
    'temp_vapor_C': 470,
    'eficiencia_termica_pct': 85,           # Fuente: Yusniati et al. (2018) - coincide con Tabla 12
    'temp_agua_alimentacion_C': 129,
}

PARAM_TURBINA_EXTRACCION_CONDENSACION = {
    'presion_vapor_proceso_bar': 5,
    'temp_vapor_proceso_C': 159,
    'eficiencia_isentropica_extraccion_pct': 77,   # Booneimsri et al. (2018)
    'eficiencia_isentropica_contrapresion_pct': 75, # para Escenario 2 (turbina contrapresión)
    'eficiencia_mecanica_pct': 98,
    'eficiencia_electrica_generador_pct': 95,
    'autoconsumo_equipo_auxiliar_pct': 14.2,        # nuevo: consumo parásito de equipos auxiliares
    'presion_condensacion_bar': 0.22,               # Montoya et al. (2020)
    'temp_condensacion_C': 62,
}

# Validación de saturación a la presión de condensación (ejecutar en tu PC con iapws)
# from iapws import IAPWS97
# sat = IAPWS97(P=0.22*0.1, x=0)
# print(f"T saturación a 0.22 bar: {sat.T - 273.15:.1f} °C  (tesis reporta: 62°C)")

### 6.4 Digestión anaeróbica — Tabla 13 y BMP de EFB pretratado
Fuente: Rahayu et al. (2015), Ocampo Batlle et al. (2020), Thomsen et al. (2014)

In [18]:
# ==========================================================
# Tabla 13 (tesis) - Sistema de digestión anaeróbica
# ==========================================================
PARAM_DIGESTION_ANAEROBICA = {
    'consumo_electricidad_kWh_Nm3_CH4': 0.227,      # Rahayu et al. (2015)
    'disponibilidad_biogas_POME_Nm3_kgDQO': 0.35,   # Rahayu et al. (2015)
    'disponibilidad_lodos_t_tEfluente': 0.03,       # Ocampo Batlle et al. (2020)
}

DQO_ENTRADA_mg_L = {
    'Planta A': 74496, 'Planta B': 97777, 'Planta C': 91573,
    'Planta D': 122973, 'Planta E': 97338, 'Planta F': 94964
}

# ==========================================================
# BMP de EFB pretratado - Thomsen et al. (2014), Bioresource Technology 154:80-86
# Validado 100% exacto contra tesis (273.51 L CH4/kg SV) usando:
# x_R = 1 - (x_C + x_H + x_L)  <- complemento matemático, NO es directamente la ceniza
# ==========================================================
def bmp_efb(x_C, x_H, x_L):
    """BMP de material lignocelulósico (L CH4/kg SV)."""
    x_R = 1 - (x_C + x_H + x_L)
    return 378*x_C + 354*x_H - 194*x_L + 313*x_R

BMP_EFB_L_CH4_kg_SV = bmp_efb(x_C=0.43, x_H=0.21, x_L=0.15)
print(f"BMP EFB: {BMP_EFB_L_CH4_kg_SV:.2f} L CH4/kg SV")  # debe dar 273.51

BMP EFB: 273.51 L CH4/kg SV


### 6.5 Pretratamiento térmico de EFB (steam explosion) — Tablas 14 y 15 (Escenario 2)
Parámetros operativos del reactor (Tabla 14) y parámetros de producción de metano 
tras el pretratamiento (Tabla 15). Fuente: J. A. Voogt et al. (2021)

In [19]:
# ==========================================================
# Tabla 14 (tesis) - Parámetros operativos del reactor de 
# pretratamiento térmico (steam explosion) - Escenario 2
# Distinto del pretratamiento mecánico (Tabla 11, Escenario 1)
# ==========================================================
PARAM_PRETRATAMIENTO_TERMICO_EFB = {
    'temp_vapor_C': 200,                        # J. A. Voogt et al. (2021)
    'presion_vapor_bar': 16,                    # J. A. Voogt et al. (2021)
    'flujo_vapor_t_vapor_t_biomasa': 0.3,       # t vapor / t biomasa (EFB)
    'recuperacion_vapor_pct': 50,               # % de vapor recuperado (flash steam)
    'consumo_electricidad_kWh_tEFB': 1,         # kWh / t EFB procesado
}

# ==========================================================
# Tabla 15 (tesis) - Parámetros de producción de metano tras 
# pretratamiento térmico
# Sección "Determinado experimentales": datos medidos directamente 
# sobre EFB pretratado (no hay valores separados de Efluente aquí)
# ==========================================================
PARAM_METANO_EFB_EXPERIMENTAL = {
    'masa_seca_pct': 44,
    'humedad_pct': 56,
    'remocion_carga_organica_pct': 66,
    'produccion_biogas_m3_tMateriaOrganica': 475,
    'incremento_produccion_biogas_pct': 36,     # vs. EFB sin pretratar
    'contenido_CH4_pct': 54,
}

# Sección "Suposiciones de diseño conceptual": sí trae Efluente y EFB
# por separado (valores de diseño, no medición directa)
PARAM_METANO_DISENO_CONCEPTUAL = {
    'Efluente': {
        'masa_seca_pct': 9,
        'humedad_pct': 91,
        'remocion_carga_organica_pct': 85,
        'produccion_biogas_m3_tMOin': 538,
        'produccion_biogas_m3_tMS': 39,
        'contenido_CH4_pct': 65,
    },
    'EFB': {
        'masa_seca_pct': 44,
        'humedad_pct': 56,
        'alta_remocion_carga_organica_pct': 10,   # solo EFB (supuesto de diseño alto)
        'remocion_carga_organica_pct': 73,
        'produccion_biogas_m3_tMOin': 523,
        'produccion_biogas_m3_tMS': 219,
        'contenido_CH4_pct': 54,
    }
}

print("Tabla 14 - Reactor pretratamiento térmico:")
for k, v in PARAM_PRETRATAMIENTO_TERMICO_EFB.items():
    print(f"  {k}: {v}")

print("\nTabla 15 - Datos experimentales (EFB):")
for k, v in PARAM_METANO_EFB_EXPERIMENTAL.items():
    print(f"  {k}: {v}")

print("\nTabla 15 - Suposiciones de diseño conceptual:")
for biomasa, params in PARAM_METANO_DISENO_CONCEPTUAL.items():
    print(f"  {biomasa}:")
    for k, v in params.items():
        print(f"    {k}: {v}")

Tabla 14 - Reactor pretratamiento térmico:
  temp_vapor_C: 200
  presion_vapor_bar: 16
  flujo_vapor_t_vapor_t_biomasa: 0.3
  recuperacion_vapor_pct: 50
  consumo_electricidad_kWh_tEFB: 1

Tabla 15 - Datos experimentales (EFB):
  masa_seca_pct: 44
  humedad_pct: 56
  remocion_carga_organica_pct: 66
  produccion_biogas_m3_tMateriaOrganica: 475
  incremento_produccion_biogas_pct: 36
  contenido_CH4_pct: 54

Tabla 15 - Suposiciones de diseño conceptual:
  Efluente:
    masa_seca_pct: 9
    humedad_pct: 91
    remocion_carga_organica_pct: 85
    produccion_biogas_m3_tMOin: 538
    produccion_biogas_m3_tMS: 39
    contenido_CH4_pct: 65
  EFB:
    masa_seca_pct: 44
    humedad_pct: 56
    alta_remocion_carga_organica_pct: 10
    remocion_carga_organica_pct: 73
    produccion_biogas_m3_tMOin: 523
    produccion_biogas_m3_tMS: 219
    contenido_CH4_pct: 54


## 7. Eficiencia de caldera (línea base) — ecuación ASME + iapws
Validación de entalpías (IAPWS97 vs. valores de la tesis) y cálculo de eficiencia térmica por planta.

In [20]:
vapor_val = IAPWS97(P=24*0.1, T=256+273.15)
agua_val = IAPWS97(P=1*0.1, T=95+273.15)
print(f"h_vapor (IAPWS97): {vapor_val.h:.1f} kJ/kg   | Excel reporta: 2897.3")
print(f"h_agua (IAPWS97): {agua_val.h:.1f} kJ/kg   | Excel reporta: 398")


h_vapor (IAPWS97): 2901.7 kJ/kg   | Excel reporta: 2897.3
h_agua (IAPWS97): 398.0 kJ/kg   | Excel reporta: 398


In [21]:
def entalpia_vapor(P_bar, T_C):
    return IAPWS97(P=P_bar*0.1, T=T_C+273.15).h

def entalpia_agua_alimentacion(T_C, P_bar=1.0):
    return IAPWS97(P=P_bar*0.1, T=T_C+273.15).h

def eficiencia_caldera_ASME(m_vapor_kg_h, P_bar, T_vapor_C, T_agua_C, energia_combustible_kWth):
    h_f = entalpia_vapor(P_bar, T_vapor_C)
    h_i = entalpia_agua_alimentacion(T_agua_C)
    energia_salida_kWth = m_vapor_kg_h * (h_f - h_i) / 3600  # kJ/h -> kW
    eta = energia_salida_kWth / energia_combustible_kWth
    return eta, h_f, h_i

# Datos línea base (hoja Eficiencia, filas 151-169) - las 6 plantas
datos_caldera_LB = {
    'Planta A': {'m_vapor': 13842,    'P_bar': 24,   'T_vapor': 256,   'T_agua': 95, 'energia_comb': 12691.45},
    'Planta B': {'m_vapor': 19056,    'P_bar': 11,   'T_vapor': 188,   'T_agua': 92, 'energia_comb': 22014.43},
    'Planta C': {'m_vapor': 14127,    'P_bar': 9.6,  'T_vapor': 182.4, 'T_agua': 80, 'energia_comb': 14251.88},
    'Planta D': {'m_vapor': 9878.4,   'P_bar': 9.2,  'T_vapor': 180,   'T_agua': 85, 'energia_comb': 12278.74},
    'Planta E': {'m_vapor': 15480,    'P_bar': 11.3, 'T_vapor': 189.1, 'T_agua': 90, 'energia_comb': 18579.14},
    'Planta F': {'m_vapor': 9720,     'P_bar': 10.3, 'T_vapor': 181.2, 'T_agua': 80, 'energia_comb': 9813.09},
}

resultados_eficiencia = []
for planta, d in datos_caldera_LB.items():
    eta, h_f, h_i = eficiencia_caldera_ASME(d['m_vapor'], d['P_bar'], d['T_vapor'], d['T_agua'], d['energia_comb'])
    resultados_eficiencia.append({'Planta': planta, 'Eficiencia_termica_pct': eta*100, 'h_f_kJkg': h_f, 'h_i_kJkg': h_i})

df_eficiencia_caldera_LB = pd.DataFrame(resultados_eficiencia)
print(df_eficiencia_caldera_LB)


     Planta  Eficiencia_termica_pct     h_f_kJkg    h_i_kJkg
0  Planta A               75.851854  2901.723409  398.030275
1  Planta B               57.850513  2791.347175  385.403703
2  Planta C               67.511090  2786.874036  334.990544
3  Planta D               54.252441  2783.645373  355.979148
4  Planta E               55.896269  2792.118503  376.991515
5  Planta F               67.226038  2778.305957  334.990544


In [22]:
df_eficiencia_caldera_LB.to_csv('eficiencia_caldera_linea_base.csv', index=False)

## 8. Estacionalidad — RFF evolution
Factor de estacionalidad mensual (2022) y buffer de almacenamiento de biomasa requerido — material para la discusión del artículo.

In [23]:
# ==========================================================
# PASO 1: Cargar datos mensuales (RFF evolution, 2022)
# ==========================================================
raw_rff = pd.read_excel('Escenarios.xlsx', sheet_name='RFF evolution', header=None)

# La tabla empieza en la fila 6 de Excel (índice 5 en pandas)
meses = raw_rff.iloc[6:18, 1].tolist()
plantas = ['Planta A', 'Planta B', 'Planta C', 'Planta D', 'Planta E', 'Planta F']

registros_rff = []
for i, mes in enumerate(meses):
    fila = 6 + i
    for j, planta in enumerate(plantas):
        col = 2 + j
        valor = raw_rff.iloc[fila, col]
        registros_rff.append({'Mes': mes, 'Mes_num': i+1, 'Planta': planta, 'RFF_ton': valor})

df_rff_mensual = pd.DataFrame(registros_rff)
df_rff_mensual['RFF_ton'] = pd.to_numeric(df_rff_mensual['RFF_ton'], errors='coerce')

print(df_rff_mensual.shape)
print(df_rff_mensual.head(15))

# ==========================================================
# PASO 2: Factor de estacionalidad (relativo al promedio mensual)
# ==========================================================
promedio_mensual = df_rff_mensual.groupby('Planta')['RFF_ton'].transform('mean')
df_rff_mensual['Factor_estacionalidad'] = df_rff_mensual['RFF_ton'] / promedio_mensual
df_rff_mensual['Excedente_deficit_ton'] = df_rff_mensual['RFF_ton'] - promedio_mensual

print(df_rff_mensual.pivot(index='Mes', columns='Planta', values='Factor_estacionalidad'))

# ==========================================================
# PASO 3: Necesidad de almacenamiento (buffer para generación constante)
# ==========================================================
def biomasa_almacenamiento_requerido(df_planta):
    df_planta = df_planta.sort_values('Mes_num')
    acumulado = df_planta['Excedente_deficit_ton'].cumsum()
    return acumulado.max() - acumulado.min()

resultados_buffer = []
for planta in plantas:
    df_p = df_rff_mensual[df_rff_mensual['Planta'] == planta]
    buffer_ton = biomasa_almacenamiento_requerido(df_p)
    total_anual = df_p['RFF_ton'].sum()
    resultados_buffer.append({
        'Planta': planta,
        'Buffer_almacenamiento_ton': buffer_ton,
        'Pct_del_total_anual': (buffer_ton / total_anual) * 100
    })

df_buffer = pd.DataFrame(resultados_buffer)
print(df_buffer)


(72, 4)
        Mes  Mes_num    Planta    RFF_ton
0     ENERO        1  Planta A   7051.000
1     ENERO        1  Planta B  25334.340
2     ENERO        1  Planta C  14053.000
3     ENERO        1  Planta D  11129.000
4     ENERO        1  Planta E   6475.319
5     ENERO        1  Planta F   3834.000
6   FEBRERO        2  Planta A   7403.000
7   FEBRERO        2  Planta B  21591.000
8   FEBRERO        2  Planta C  14614.000
9   FEBRERO        2  Planta D  11764.000
10  FEBRERO        2  Planta E   7139.000
11  FEBRERO        2  Planta F   3506.550
12    MARZO        3  Planta A  18240.923
13    MARZO        3  Planta B  21943.110
14    MARZO        3  Planta C  17161.000
Planta      Planta A  Planta B  Planta C  Planta D  Planta E  Planta F
Mes                                                                   
ABRIL       1.284151  1.198443  1.091081  1.025341  1.111447  1.094356
AGOSTO      0.923904  0.680771  0.976339  0.859863  0.895418  0.819680
DICIEMBRE   1.115671  0.600619  0.90

In [24]:
df_rff_mensual.to_csv('rff_mensual_estacionalidad.csv', index=False)
df_buffer.to_csv('buffer_almacenamiento_biomasa.csv', index=False)

### 8.1 Cogeneración existente en línea base — fuente del consumo eléctrico
Desglose por planta de la fracción de electricidad autogenerada (biomasa/biogás) 
vs. comprada (red nacional/diésel). Fuente: hoja Eficiencia, filas 220-223.

In [25]:
# ==========================================================
# Fuente del consumo eléctrico por planta - cogeneración existente 
# en línea base (hoja Eficiencia, filas 220-223)
# Dato reportado directamente (sin fórmula intermedia) -> trazable
# ==========================================================
raw_efi = pd.read_excel('Escenarios.xlsx', sheet_name='Eficiencia', header=None)

df_fuente_electrica_LB = pd.DataFrame({
    'Planta': plantas,  # ya definido en secciones anteriores
    'Consumo_electrico_total_kWh_ano': raw_efi.iloc[219, 2:8].tolist(),
    'Frac_biomasa_cogeneracion': raw_efi.iloc[220, 2:8].tolist(),
    'Frac_biogas': raw_efi.iloc[221, 2:8].tolist(),
    'Frac_red_nacional': raw_efi.iloc[222, 2:8].tolist(),
    'Frac_diesel': raw_efi.iloc[223, 2:8].tolist()
})

# Electricidad autogenerada por cada fuente (kWh/año)
df_fuente_electrica_LB['Electricidad_biomasa_kWh_ano'] = (
    df_fuente_electrica_LB['Consumo_electrico_total_kWh_ano'] * df_fuente_electrica_LB['Frac_biomasa_cogeneracion'])
df_fuente_electrica_LB['Electricidad_biogas_kWh_ano'] = (
    df_fuente_electrica_LB['Consumo_electrico_total_kWh_ano'] * df_fuente_electrica_LB['Frac_biogas'])
df_fuente_electrica_LB['Electricidad_red_kWh_ano'] = (
    df_fuente_electrica_LB['Consumo_electrico_total_kWh_ano'] * df_fuente_electrica_LB['Frac_red_nacional'])
df_fuente_electrica_LB['Electricidad_diesel_kWh_ano'] = (
    df_fuente_electrica_LB['Consumo_electrico_total_kWh_ano'] * df_fuente_electrica_LB['Frac_diesel'])

# Validación de consistencia: las fracciones deben sumar 1.0 en cada planta
suma_check = df_fuente_electrica_LB[['Frac_biomasa_cogeneracion','Frac_biogas',
                                       'Frac_red_nacional','Frac_diesel']].sum(axis=1)
assert (suma_check.round(6) == 1.0).all(), "Las fracciones no suman 1.0 - revisar datos"

print("Solo Planta A tiene cogeneración con biomasa; Planta D ya tiene biogás (70%):")
print(df_fuente_electrica_LB[['Planta','Frac_biomasa_cogeneracion','Frac_biogas',
                                'Electricidad_biomasa_kWh_ano']])

df_fuente_electrica_LB.to_csv('fuente_electrica_linea_base.csv', index=False)

Solo Planta A tiene cogeneración con biomasa; Planta D ya tiene biogás (70%):
     Planta  Frac_biomasa_cogeneracion  Frac_biogas  \
0  Planta A                   0.791279          0.0   
1  Planta B                   0.000000          0.0   
2  Planta C                   0.000000          0.0   
3  Planta D                   0.000000          0.7   
4  Planta E                   0.000000          0.0   
5  Planta F                   0.000000          0.0   

   Electricidad_biomasa_kWh_ano  
0                  2.440092e+06  
1                  0.000000e+00  
2                  0.000000e+00  
3                  0.000000e+00  
4                  0.000000e+00  
5                  0.000000e+00  


## 9. Modelo Pyomo — Línea base (prueba de concepto, Planta A)
Balance de energía térmica (caldera), electricidad (autogenerada + red + diésel) 
y emisiones de GEI. Usa datos ya limpios de las secciones anteriores — 
correr después de haber ejecutado 1, 6, 7, 8 y 8.1.

In [26]:
# ==========================================================
# Modelo Pyomo - Línea base, Planta A (prueba de concepto, Fase 1)
# Fase 1: capacidad fija, sin optimizar - el modelo verifica que 
# el balance de energía y electricidad cierra con datos medidos.
# La Fase 2 (futura) agregará variables de decisión reales.
# ==========================================================

# --- Datos de entrada para Planta A (de secciones ya ejecutadas) ---
d_caldera = datos_caldera_LB['Planta A']          # Sección 7
eta_caldera_A = df_eficiencia_caldera_LB.loc[
    df_eficiencia_caldera_LB['Planta']=='Planta A', 'Eficiencia_termica_pct'].iloc[0] / 100

fila_electrica_A = df_fuente_electrica_LB[df_fuente_electrica_LB['Planta']=='Planta A'].iloc[0]

model = pyo.ConcreteModel(name="Linea_Base_Planta_A")

# --- Parámetros (datos fijos, medidos) ---
model.m_vapor_kg_h        = pyo.Param(initialize=d_caldera['m_vapor'])
model.energia_comb_kWth   = pyo.Param(initialize=d_caldera['energia_comb'])
model.eta_caldera         = pyo.Param(initialize=eta_caldera_A)
model.consumo_elec_total  = pyo.Param(initialize=fila_electrica_A['Consumo_electrico_total_kWh_ano'])
model.elec_biomasa_medida = pyo.Param(initialize=fila_electrica_A['Electricidad_biomasa_kWh_ano'])
model.elec_red_medida     = pyo.Param(initialize=fila_electrica_A['Electricidad_red_kWh_ano'])
model.elec_diesel_medida  = pyo.Param(initialize=fila_electrica_A['Electricidad_diesel_kWh_ano'])

# --- Variables (en Fase 1 quedan completamente determinadas por las restricciones) ---
model.energia_util_vapor_kWth = pyo.Var(domain=pyo.NonNegativeReals)   # energía térmica útil entregada por el vapor
model.electricidad_biomasa    = pyo.Var(domain=pyo.NonNegativeReals)
model.electricidad_red        = pyo.Var(domain=pyo.NonNegativeReals)
model.electricidad_diesel     = pyo.Var(domain=pyo.NonNegativeReals)

# --- Restricciones ---
# 1) Energía térmica útil = eficiencia de caldera * energía de combustible
model.balance_termico = pyo.Constraint(
    expr=model.energia_util_vapor_kWth == model.eta_caldera * model.energia_comb_kWth
)

# 2) Balance eléctrico: cada fuente debe igualar el dato medido
#    (en Fase 1 no hay grados de libertad; en Fase 2 esto se relaja)
model.def_elec_biomasa = pyo.Constraint(expr=model.electricidad_biomasa == model.elec_biomasa_medida)
model.def_elec_red     = pyo.Constraint(expr=model.electricidad_red == model.elec_red_medida)
model.def_elec_diesel  = pyo.Constraint(expr=model.electricidad_diesel == model.elec_diesel_medida)

# 3) Restricción de cierre: la suma de fuentes debe igualar el consumo total
model.cierre_electrico = pyo.Constraint(
    expr=model.electricidad_biomasa + model.electricidad_red + model.electricidad_diesel
         == model.consumo_elec_total
)

# --- Objetivo dummy (Fase 1 no optimiza, solo valida el balance) ---
model.obj = pyo.Objective(expr=1, sense=pyo.minimize)

# --- Resolver ---
solver = pyo.SolverFactory('glpk')
resultado = solver.solve(model, tee=False)

print("Estado del solver:", resultado.solver.termination_condition)
print(f"\nEnergía térmica útil (vapor):  {pyo.value(model.energia_util_vapor_kWth):,.1f} kWth")
print(f"Electricidad autogenerada (biomasa): {pyo.value(model.electricidad_biomasa):,.1f} kWh/año")
print(f"Electricidad de red nacional:        {pyo.value(model.electricidad_red):,.1f} kWh/año")
print(f"Electricidad de diésel:              {pyo.value(model.electricidad_diesel):,.1f} kWh/año")
print(f"Consumo eléctrico total (verificación): {pyo.value(model.electricidad_biomasa + model.electricidad_red + model.electricidad_diesel):,.1f} kWh/año")

Estado del solver: optimal

Energía térmica útil (vapor):  9,626.7 kWth
Electricidad autogenerada (biomasa): 2,440,092.0 kWh/año
Electricidad de red nacional:        609,714.6 kWh/año
Electricidad de diésel:              33,924.4 kWh/año
Consumo eléctrico total (verificación): 3,083,731.0 kWh/año


## 10. Modelo Pyomo — Escenario 1 (turbina extracción-condensación)
*(pendiente)*

## 11. Modelo Pyomo — Escenario 2 (turbina contrapresión + biogás)
*(pendiente)*